In [ ]:
import pandas as pd
import os
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
from torchvision import models
import torch.nn as nn
from torch.utils.data import ConcatDataset
import torch.optim as optim

dataset : https://www.kaggle.com/datasets/puneet6060/intel-image-classification?select=seg_train

Create the label files

In [ ]:
dataset_train_path = '/Users/laurageneaulibourne/Downloads/S4/ia/exam/dataset/seg_train/seg_train'
dataset_test_path = '/Users/laurageneaulibourne/Downloads/S4/ia/exam/dataset/seg_test/seg_test'

classe = {'buildings' : 0, 'forest' : 1, 'glacier' : 2, 'mountain' : 3, 'sea' : 4, 'street' : 5}

for c in classe.keys():
    label_train = []
    label_test = []

    class_train_path = os.path.join(dataset_train_path, c)
    class_test_path = os.path.join(dataset_test_path, c)

    img_train_path = class_train_path +'/img'
    img_test_path = class_test_path+'/img'

    for imag in os.listdir(img_train_path):
        if not imag.startswith('.'): # avoid .DS_Store
            label_train.append({'image_name' : imag, 'label' : classe[c]})


    for imag in os.listdir(img_test_path):
        if not imag.startswith('.'):
            label_test.append({'image_name' : imag, 'label' : classe[c]})


    df_train = pd.DataFrame(label_train)
    df_train.to_csv(os.path.join(class_train_path, f"{classe[c]}_label_train.csv"), index=False)
    df_test = pd.DataFrame(label_test)
    df_test.to_csv(os.path.join(class_test_path, f"{classe[c]}_label_test.csv"), index=False)



create our dataset

In [ ]:
#For each task we will do a classification on 2 classes, so we will create a new dataset with only 2 classes by merging our 2 directories and same thing for the file with the labels.

def task(A,B, idx):
    '''A and B are the 2 classes of the idx task'''
    path_img_A_train = os.path.join(dataset_train_path, A, 'img')
    path_img_B_train = os.path.join(dataset_train_path, B, 'img')
    path_img_AB_train = os.path.join(dataset_train_path, f'df_{A}_{B}', 'img')
    path_img_A_test = os.path.join(dataset_test_path, A, 'img')
    path_img_B_test = os.path.join(dataset_test_path, B, 'img')
    path_img_AB_test = os.path.join(dataset_test_path, f'df_{A}_{B}', 'img')

    #create the directory if it does not exist
    os.makedirs(path_img_AB_train, exist_ok=True)
    os.makedirs(path_img_AB_test, exist_ok=True)

    def move(init, final):
        for file in os.listdir(init):
            if not file.startswith('.'):
                source_file = os.path.join(init, file)
                dest_file = os.path.join(final, file)
                os.rename(source_file, dest_file)



    move(path_img_A_train, path_img_AB_train)
    move(path_img_B_train, path_img_AB_train)

    move(path_img_A_test, path_img_AB_test)
    move(path_img_B_test, path_img_AB_test)

    label_A_train = pd.read_csv(os.path.join(dataset_train_path, A, f"{classe[A]}_label_train.csv"))
    label_B_train = pd.read_csv(os.path.join(dataset_train_path, B, f"{classe[B]}_label_train.csv"))
    label_A_train['label'] = 0
    label_B_train['label'] = 1
    label_AB_train = pd.concat([label_A_train, label_B_train], ignore_index=True)
    label_AB_train['task_id'] = idx
    label_AB_train.to_csv(os.path.join(dataset_train_path,f'df_{A}_{B}', 'label_train.csv'), index=False)



    label_A_test = pd.read_csv(os.path.join(dataset_test_path, A, f"{classe[A]}_label_test.csv"))
    label_B_test = pd.read_csv(os.path.join(dataset_test_path, B, f"{classe[B]}_label_test.csv"))
    label_A_test['label'] = 0
    label_B_test['label'] = 1
    label_AB_test = pd.concat([label_A_test, label_B_test], ignore_index=True)
    label_AB_test['task_id'] = idx
    label_AB_test.to_csv(os.path.join(dataset_test_path, f'df_{A}_{B}', 'label_test.csv'), index=False)

    return (path_img_AB_train, path_img_AB_test, label_AB_train, label_AB_test)

path_img_AB_train, path_img_AB_test, label_AB_train, label_AB_test = task('buildings', 'sea', 0)
path_img_CD_train, path_img_CD_test, label_CD_train, label_CD_test = task('forest', 'mountain', 1)

Repartition of the dataset through the 6 classes

In [ ]:

def repartition(label_train, label_test, classe_0, classe_1):

    train_tot = len(label_train) # total number of images in the train for this task
    test_tot  = len(label_test) # total number of images in the test for this task

    nb_0_train = (label_train['label'] == 0).sum() #number of images in the train for the class 0
    nb_1_train = (label_train['label'] == 1).sum() #number of images in the train for the class 1
    nb_0_test  = (label_test['label'] == 0).sum() #number of images in the test for the class 0
    nb_1_test  = (label_test['label'] == 1).sum() #number of images in the test for the class 1

    print(f"Task : {label_train['task_id'].iloc[0]} \n")
    print(f" Total images in the train for the task {classe_0}/{classe_1} : {train_tot}")
    print(f" Pourcentage of images in the train for the class {classe_0} : {nb_0_train * 100/ train_tot}%")
    print(f" Pourcentage of images in the train for the class {classe_1} : {nb_1_train * 100/ train_tot}%")

    print(f" Total images in the test for the task {classe_0}/{classe_1} : {test_tot}")
    print(f" Pourcentage of images in the test for the class {classe_0} : {nb_0_test * 100/ test_tot}%")
    print(f" Pourcentage of images in the test for the class {classe_1} : {nb_1_test * 100/ test_tot}% \n")


repartition(label_AB_train, label_AB_test, 'buildings', 'sea')
repartition(label_CD_train, label_CD_test, 'forest', 'mountain')



Task : 0 

 Total images in the train for the task buildings/sea : 4465
 Pourcentage of images in the train for the class buildings : 49.07054871220605%
 Pourcentage of images in the train for the class sea : 50.92945128779395%
 Total images in the test for the task buildings/sea : 947
 Pourcentage of images in the test for the class buildings : 46.14572333685322%
 Pourcentage of images in the test for the class sea : 53.85427666314678% 

Task : 1 

 Total images in the train for the task forest/mountain : 4783
 Pourcentage of images in the train for the class forest : 47.48066067321765%
 Pourcentage of images in the train for the class mountain : 52.51933932678235%
 Total images in the test for the task forest/mountain : 999
 Pourcentage of images in the test for the class forest : 47.447447447447445%
 Pourcentage of images in the test for the class mountain : 52.552552552552555% 



In [ ]:
"""

def portion_dataset(images,label, portion): #returns a portion of the dataset (images and labels) given a portion value between 0 and 1.

    nb = int(len(label) * portion) #number of images in the dataset that we will keep in order to avoid catastrophic forgetting during the new task.

    idx = np.random.choice(len(label), size=nb, replace=False) #indexes of the images that we will keep.

    img_portion = images[idx]
    label_portion = label[idx]

    return img_portion, label_portion

"""

In [ ]:
class CustomImageDataset(Dataset):

    def __init__(self, annotations_file, img_dir, portion, img_transform=None): # portion is the pourcentage of the dataset that we will keep in order to avoid catastrophic forgetting during the new task.

        self.img_dir = img_dir
        self.img_transform = img_transform

        if portion < 1.0:
            self.img_labels = annotations_file.sample(frac=portion, random_state=42).reset_index(drop=True)
        else:
            self.img_labels = annotations_file

    def __len__(self): #Return the number of samples in the dataset.
        return len(self.img_labels)

    def __getitem__(self, idx): #For a given index, it returns the sample (image, label) associated to it.

        # Get the file path of the image by combining the directory and the image filename from the labels DataFrame
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])

        # Read the image from the specified file path and convert it to a tensor
        image = Image.open(img_path)

        # Retrieve the corresponding label for the image from the labels DataFrame
        label = self.img_labels.iloc[idx, 1]

        task = self.img_labels.iloc[idx, 2]

        # If a transform function is provided, apply it to the image
        if self.img_transform:
            image = self.img_transform(image)
            # If a target_transform function is provided, apply it to the label

        # Return the transformed image and label as a tuple
        return image, label, task


In [ ]:
# List of transformations that will be applied to images
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])


In [ ]:

dataset_train_AB_full = CustomImageDataset(annotations_file= label_AB_train, img_dir = path_img_AB_train, portion = 1 , img_transform=transform)
dataset_test_AB_full = CustomImageDataset(annotations_file= label_AB_test, img_dir = path_img_AB_test, portion = 1 , img_transform=transform)

train_dataloader_AB_full = DataLoader(dataset_train_AB_full, batch_size=64, shuffle=True)
test_dataloader_AB_full= DataLoader(dataset_test_AB_full, batch_size=64, shuffle=True)


portion = 0.5 #portion of the dataset of the previous dataset we want to keep for avoiding the catastrophic forgetting during the new task.



dataset_train_AB_part = CustomImageDataset(annotations_file= label_AB_train, img_dir = path_img_AB_train, portion = portion , img_transform=transform)
dataset_test_AB_part = CustomImageDataset(annotations_file= label_AB_test, img_dir = path_img_AB_test, portion = portion, img_transform=transform)

train_dataloader_AB_part = DataLoader(dataset_train_AB_part, batch_size=64, shuffle=True)
test_dataloader_AB_part= DataLoader(dataset_test_AB_part, batch_size=64,  shuffle=True)





dataset_train_CD_full = CustomImageDataset(annotations_file= label_CD_train, img_dir = path_img_CD_train, portion = 1 , img_transform=transform)
dataset_test_CD_full = CustomImageDataset(annotations_file= label_CD_test, img_dir = path_img_CD_test, portion = 1, img_transform=transform)

train_dataloader_CD_full = DataLoader(dataset_train_CD_full, batch_size=64, shuffle=True)
test_dataloader_CD_full= DataLoader(dataset_test_CD_full, batch_size=64, shuffle=True)






dataset_train_ABCD = ConcatDataset([dataset_train_AB_part, dataset_train_CD_full])
dataset_test_ABCD = ConcatDataset([dataset_test_AB_part, dataset_test_CD_full])

train_dataloader_ABCD = DataLoader(dataset_train_ABCD, batch_size=64, shuffle=True)
test_dataloader_ABCD= DataLoader(dataset_test_ABCD, batch_size=64, shuffle=True)


In [ ]:

"""
def reduce_dataset(dataset, portion):
    # portion = the portion of the dataset of the previous tasks we want to keep for avoiding the catastrophic forgetting during the new task.
    nb = int(len(dataset) * portion) #number of images to keep

    # 2. Générer des indices aléatoires uniques
    indices = np.random.choice(len(dataset), size=nb, replace=False)
    return Subset(dataset, indices)

# 3. Créer le sous-jeu de données et le passer au DataLoader standard
dataset_reduit = Subset(dataset_train, indices)
train_loader = DataLoader(dataset_reduit, batch_size=64, shuffle=True)

"""

In [ ]:
basic_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = basic_model.fc.in_features
basic_model.fc = nn.Linear(num_features, 2)

In [ ]:
def train_loop_basic(dataloader, model, loss, optimizer, num_epochs, device):
    size = len(dataloader.dataset)
    model = model.to(device)
    model.train()

    for epoch in range(num_epochs):

        epoch_loss = 0.0
        dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0}  # accuracy dictionary for each class

        for image, label, _ in dataloader:

            #put image and label on the same device
            image= image.to(device)
            label= label.to(device)

            optimizer.zero_grad()  # Set gradient to zero

            # Compute prediction and loss
            pred = model(image)

            batch_loss = loss(pred, label)
            epoch_loss += batch_loss.item()

            pred_class = pred.argmax(dim=1)
            pred_classA = (pred_class == 0)
            label_A = (label == 0)
            dic_acc["A_true"] += (pred_classA & label_A).sum().item()
            dic_acc["A_tot"] += label_A.sum().item()

            pred_classB = (pred_class == 1)
            label_B = (label == 1)
            dic_acc["B_true"] += (pred_classB & label_B).sum().item()
            dic_acc["B_tot"] += label_B.sum().item()

            # Backpropagation
            batch_loss.backward()
            optimizer.step()

        epoch_loss = epoch_loss / size


        print(f'Epoch {epoch+1}/{num_epochs}, Total Mean Loss: {epoch_loss}, '
              f'Accuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0)} %, '
              f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0)} %, '
    )




In [ ]:
def test_loop_basic(dataloader, model, loss, device):
    size = len(dataloader.dataset)
    model = model.to(device)
    model.eval()

    with torch.no_grad():

        test_loss = 0.0
        dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0}  # accuracy dictionary for each class

        for image, label, _ in dataloader:

            #put image and label on the same device
            image= image.to(device)
            label= label.to(device)

            # Compute prediction and loss
            pred = model(image)

            batch_loss = loss(pred, label)
            test_loss += batch_loss.item()* label.size(0)

            pred_class = pred.argmax(dim=1)
            pred_classA = (pred_class == 0)
            label_A = (label == 0)
            dic_acc["A_true"] += (pred_classA & label_A).sum().item()
            dic_acc["A_tot"] += label_A.sum().item()

            pred_classB = (pred_class == 1)
            label_B = (label == 1)
            dic_acc["B_true"] += (pred_classB & label_B).sum().item()
            dic_acc["B_tot"] += label_B.sum().item()

            # Backpropagation
            batch_loss.backward()
            optimizer.step()

        test_loss = test_loss / size


        print(f'Test Loss: {test_loss}, '
              f'Accuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0)} %, '
              f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0)} %, '
    )




In [ ]:
class rehearsal(nn.Module):
# Constructor method to initialize layers of the network def __init__(self):
        def __init__(self,num_classe=2):
                super().__init__()
                self.num_classe = num_classe
                self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT) # we consider a pretrained model
                self.num_features = self.model.fc.in_features
                self.model.fc = nn.Identity()
                self.heads = nn.ModuleDict()

        def add_task(self, num_task):
                num_task = str(num_task)
                if num_task not in self.heads:
                        self.heads[num_task] = nn.Linear(self.num_features, self.num_classe)

        # Forward pass of the network
        def forward(self, x, num_task):
                features = self.model(x)  # features pour toutes les images du batch
                outputs = torch.zeros(x.size(0), self.num_classe, device=x.device)

                for t in torch.unique(num_task):
                        t_str = str(t.item())
                        if t_str not in self.heads:
                                raise ValueError(f"Task {t_str} not found in the model.")
                        mask = (num_task == t)
                        outputs[mask] = self.heads[t_str](features[mask])

                return outputs

'''

        def forward(self, x, num_task):
                x = self.model(x)
                num_task = num_task[0].item() if isinstance(num_task, torch.Tensor) else num_task
                num_task = str(num_task)
                if num_task in self.heads:
                        return self.heads[num_task](x)
                else:
                        raise ValueError(f"Task {num_task} not found in the model.")
'''

'    \n        \n        def forward(self, x, num_task):\n                x = self.model(x)\n                num_task = num_task[0].item() if isinstance(num_task, torch.Tensor) else num_task\n                num_task = str(num_task)  \n                if num_task in self.heads:\n                        return self.heads[num_task](x)\n                else:\n                        raise ValueError(f"Task {num_task} not found in the model.")\n'

In [ ]:
def train_loop_rehearsal(dataloader, model, loss, optimizer, num_epochs, device):

    model = model.to(device)
    model.train()

    for epoch in range(num_epochs):

        epoch_lossAB = 0.0
        epoch_lossCD = 0.0
        epoch_loss_tot = 0.0
        dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0,  "C_true": 0, "C_tot" : 0, "D_true": 0, "D_tot": 0}  # accuracy dictionary for each class


        for image, label, task in dataloader:


            batch_lossAB = 0.0
            batch_lossCD = 0.0
            batch_loss_tot = 0.0
            #put image and label on the same device
            image= image.to(device)
            label= label.to(device)

            optimizer.zero_grad()  # Set gradient to zero

            # Compute prediction and loss
            pred = model(image, task)

            mask_AB = (task == 0)
            mask_CD = (task == 1)

            if mask_AB.sum() > 0: # at least one image from the 1st task in the batch
                pred_AB = pred[mask_AB]
                label_AB = label[mask_AB]

                batch_lossAB = loss(pred_AB, label_AB)
                epoch_lossAB += batch_lossAB.item()* label_AB.size(0)

                # For each image, choose the class with the higher score
                pred_classAB = pred_AB.argmax(dim=1)
                pred_classA = (pred_classAB == 0)
                label_A = (label_AB == 0)
                dic_acc["A_true"] += (pred_classA & label_A).sum().item()
                dic_acc["A_tot"] += label_A.sum().item()

                pred_classB = (pred_classAB == 1)
                label_B = (label_AB == 1)
                dic_acc["B_true"] += (pred_classB & label_B).sum().item()
                dic_acc["B_tot"] += label_B.sum().item()


            if mask_CD.sum() > 0: # at least one image from the 2nd task in the batch
                pred_CD = pred[mask_CD]
                label_CD = label[mask_CD]

                batch_lossCD = loss(pred_CD, label_CD)
                epoch_lossCD += batch_lossCD.item()* label_CD.size(0)

                pred_classCD = pred_CD.argmax(dim=1)
                pred_classC = (pred_classCD == 0)
                label_C = (label_CD == 0)
                dic_acc["C_true"] += (pred_classC & label_C).sum().item()
                dic_acc["C_tot"] += label_C.sum().item()

                pred_classD = (pred_classCD == 1)
                label_D = (label_CD == 1)
                dic_acc["D_true"] += (pred_classD & label_D).sum().item()
                dic_acc["D_tot"] += label_D.sum().item()


            batch_loss_tot = batch_lossAB + batch_lossCD
            epoch_loss_tot += batch_loss_tot.item()

            # Backpropagation
            batch_loss_tot.backward()
            optimizer.step()

        nb_AB = dic_acc["A_tot"] + dic_acc["B_tot"]
        nb_CD = dic_acc["C_tot"] + dic_acc["D_tot"]

        epoch_lossAB = epoch_lossAB / nb_AB
        if nb_CD > 0:
            epoch_lossCD = epoch_lossCD / nb_CD
        else :
            epoch_lossCD = 0.0
        epoch_loss = epoch_lossAB + epoch_lossCD


        print(f'Epoch {epoch+1}/{num_epochs}, Total Mean Loss: {epoch_loss}, 1st task Loss: {epoch_lossAB}, 2nd task Loss: {epoch_lossCD}, '
              f'Accuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0)} %, '
              f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0)} %, '
              f'C: {(dic_acc["C_true"] * 100 / dic_acc["C_tot"] if dic_acc["C_tot"] > 0 else 0.0)} %, '
              f'D: {(dic_acc["D_true"] * 100 / dic_acc["D_tot"] if dic_acc["D_tot"] > 0 else 0.0)} %')




In [ ]:
def test_loop_rehearsal(dataloader, model, loss, device):

    model = model.to(device)
    model.eval()
    loss_test = 0.0
    loss_testAB = 0.0
    loss_testCD = 0.0

    dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0,  "C_true": 0, "C_tot" : 0, "D_true": 0, "D_tot": 0}  # accuracy dictionary for each class

    with torch.no_grad():


        for image, label, task in dataloader:

            batch_lossAB = 0.0
            batch_lossCD = 0.0
            batch_loss_tot = 0.0

            #put image and label on the same device
            image= image.to(device)
            label= label.to(device)

            # Compute prediction and loss
            pred = model(image, task)

            mask_AB = (task == 0)
            mask_CD = (task == 1)

            if mask_AB.sum() > 0: # at least one image from the 1st task in the batch
                pred_AB = pred[mask_AB]
                label_AB = label[mask_AB]

                batch_lossAB = loss(pred_AB, label_AB)
                loss_testAB += batch_lossAB.item()* label_AB.size(0)

                # For each image, choose the class with the higher score
                pred_classAB = pred_AB.argmax(dim=1)
                pred_classA = (pred_classAB == 0)
                label_A = (label_AB == 0)
                dic_acc["A_true"] += (pred_classA & label_A).sum().item()
                dic_acc["A_tot"] += label_A.sum().item()

                pred_classB = (pred_classAB == 1)
                label_B = (label_AB == 1)
                dic_acc["B_true"] += (pred_classB & label_B).sum().item()
                dic_acc["B_tot"] += label_B.sum().item()



            if mask_CD.sum() > 0: # at least one image from the 2nd task in the batch
                pred_CD = pred[mask_CD]
                label_CD = label[mask_CD]

                batch_lossCD = loss(pred_CD, label_CD)
                loss_testCD += batch_lossCD.item()* label_CD.size(0)

                pred_classCD = pred_CD.argmax(dim=1)
                pred_classC = (pred_classCD == 0)
                label_C = (label_CD == 0)
                dic_acc["C_true"] += (pred_classC & label_C).sum().item()
                dic_acc["C_tot"] += label_C.sum().item()

                pred_classD = (pred_classCD == 1)
                label_D = (label_CD == 1)
                dic_acc["D_true"] += (pred_classD & label_D).sum().item()
                dic_acc["D_tot"] += label_D.sum().item()

        nb_AB = dic_acc["A_tot"] + dic_acc["B_tot"]
        nb_CD = dic_acc["C_tot"] + dic_acc["D_tot"]

        loss_testAB = loss_testAB / nb_AB
        if nb_CD > 0:
            loss_testCD = loss_testCD / nb_CD
        else:
            loss_testCD = 0.0

        loss_test =  loss_testAB + loss_testCD


        print(f'Total Mean Loss per images: {loss_test}, 1st task Loss: {loss_testAB}, 2nd task Loss: {loss_testCD}',
              f'Accuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0)} %, '
              f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0)} %, '
              f'C: {(dic_acc["C_true"] * 100 / dic_acc["C_tot"] if dic_acc["C_tot"] > 0 else 0.0)} %, '
              f'D: {(dic_acc["D_true"] * 100 / dic_acc["D_tot"] if dic_acc["D_tot"] > 0 else 0.0)} %')




In [ ]:
model_rehearsal = rehearsal()
model_rehearsal.add_task(0)
model_rehearsal.add_task(1)

In [ ]:
torch.backends.mps.is_available()

True

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

In [ ]:
loss = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_rehearsal.parameters(), lr=0.001, momentum=0.9)
num_epochs = 10

In [ ]:
train_loop_rehearsal(train_dataloader_AB_full , model_rehearsal, loss, optimizer, num_epochs, device)

Epoch 1/10, Total Mean Loss: 0.18114317949957237, 1st task Loss: 0.18114317949957237, 2nd task Loss: 0.0, Accuracy A: 92.05842081241443 %, B: 92.17238346525946 %, C: 0.0 %, D: 0.0 %
Epoch 2/10, Total Mean Loss: 0.027526604307008217, 1st task Loss: 0.027526604307008217, 2nd task Loss: 0.0, Accuracy A: 99.49794614331356 %, B: 99.2084432717678 %, C: 0.0 %, D: 0.0 %
Epoch 3/10, Total Mean Loss: 0.01897457836556675, 1st task Loss: 0.01897457836556675, 2nd task Loss: 0.0, Accuracy A: 99.54358740301232 %, B: 99.38434476693052 %, C: 0.0 %, D: 0.0 %
Epoch 4/10, Total Mean Loss: 0.012535999549296241, 1st task Loss: 0.012535999549296241, 2nd task Loss: 0.0, Accuracy A: 99.8630762209037 %, B: 99.8240985048373 %, C: 0.0 %, D: 0.0 %
Epoch 5/10, Total Mean Loss: 0.007907050435638842, 1st task Loss: 0.007907050435638842, 2nd task Loss: 0.0, Accuracy A: 99.8630762209037 %, B: 99.95602462620933 %, C: 0.0 %, D: 0.0 %
Epoch 6/10, Total Mean Loss: 0.008205096554407414, 1st task Loss: 0.008205096554407414, 

In [ ]:
test_loop_rehearsal(test_dataloader_AB_full, model_rehearsal, loss, device)

Total Mean Loss per images: 0.016067508288483117, 1st task Loss: 0.016067508288483117, 2nd task Loss: 0.0 Accuracy A: 99.54233409610984 %, B: 99.41176470588235 %, C: 0.0 %, D: 0.0 %


In [ ]:
train_loop_rehearsal(train_dataloader_ABCD , model_rehearsal, loss, optimizer, num_epochs, device)

Epoch 1/10, Total Mean Loss: 0.08901499142016442, 1st task Loss: 0.005147421530669673, 2nd task Loss: 0.08386756988949474, Accuracy A: 100.0 %, B: 99.73637961335676 %, C: 96.60942316160282 %, D: 97.53184713375796 %
Epoch 2/10, Total Mean Loss: 0.021413784622962524, 1st task Loss: 0.003376521601888509, 2nd task Loss: 0.018037263021074015, Accuracy A: 100.0 %, B: 99.82425307557118 %, C: 99.1193306913254 %, D: 99.8407643312102 %
Epoch 3/10, Total Mean Loss: 0.011933158379217776, 1st task Loss: 0.0009449114101134845, 2nd task Loss: 0.010988246969104292, Accuracy A: 100.0 %, B: 100.0 %, C: 99.47159841479524 %, D: 99.96019108280255 %
Epoch 4/10, Total Mean Loss: 0.007936676764109963, 1st task Loss: 0.0011202913856811969, 2nd task Loss: 0.006816385378428765, Accuracy A: 100.0 %, B: 100.0 %, C: 99.77983267283135 %, D: 100.0 %
Epoch 5/10, Total Mean Loss: 0.006206547326250809, 1st task Loss: 0.0007172986811530145, 2nd task Loss: 0.005489248645097795, Accuracy A: 100.0 %, B: 100.0 %, C: 99.77983

In [ ]:
test_loop_rehearsal(test_dataloader_ABCD, model_rehearsal, loss, device)

Total Mean Loss per images: 0.022609890823493378, 1st task Loss: 0.011621779697753302, 2nd task Loss: 0.010988111125740075 Accuracy A: 99.57627118644068 %, B: 99.15966386554622 %, C: 99.78902953586498 %, D: 99.42857142857143 %


In [ ]:
### A partir d'ici c'set la partie 2 du cod . Avant ici c'est la partoe 1 du code . La partie 2 se foculise ecxcluisiovement sur les methodes d'entrainement eavec A-GEM . La partie 1 se focalise sur les prerequis pour la constructui des datstatset , dataloader et autrse et surtout sur le continual leanring sans A -GEM

In [ ]:
import os
import torch
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image

class IntelDataset(Dataset):
    def __init__(self, df, transform=None, base_dir=None, label_map=None):
        """
        Args:
            df: DataFrame containing 'image_name' and 'label'.
            transform: Image transformations.
            base_dir: Path to the task-specific image folder.
            label_map: Mapping dictionary (global_map).
        """
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.base_dir = base_dir
        self.label_map = label_map

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['image_name']
        label = int(row['label'])

        # Remap the label if a dictionary is provided
        if self.label_map is not None:
            label = self.label_map[label]

        # Construct path: images are located in base_dir/image_name
        img_path = os.path.join(self.base_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
NB_EPOCH = 3
epochs_per_task = 6
memory_size_per_task = 300
batch_size = 32

# Detect available device
if not 'device' in locals():
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

# List of tasks to process
list_of_task = [0, 1]

# Mapping tasks to original class indices
tasks_definition = {
    0: [0, 4], # Task 0: buildings and sea
    1: [1, 3]  # Task 1: forest and mountain
}

# Mapping original labels to contiguous output neurons
global_map = {
    0: 0,  # buildings
    4: 1,  # sea
    1: 2,  # forest
    3: 3   # mountain
}

# Registry linking task IDs to existing data structures
task_data_registry = {
    0: {
        'train_df': label_AB_train,
        'test_df': label_AB_test,
        'train_path': path_img_AB_train,
        'test_path': path_img_AB_test
    },
    1: {
        'train_df': label_CD_train,
        'test_df': label_CD_test,
        'train_path': path_img_CD_train,
        'test_path': path_img_CD_test
    }
}

# Note: Training will use the data and paths from this registry.

In [ ]:
'''
NB_EPOCH=3
epochs_per_task=6 # Reduced to compensate for memory increase
memory_size_per_task=300 # We increase memory for more stability
batch_size=32

classe = {'buildings' : 0, 'forest' : 1, 'glacier' : 2, 'mountain' : 3, 'sea' : 4, 'street' : 5}

device="cuda" if torch.cuda.is_available() else "cpu"

# Task definition
list_of_task=[0,1]

# remapping to avoid problems
global_map = {
    0: 0,  # buildings -> neuron 0
    1: 1,  # forest    -> neuron 1
    3: 2,   # mountain  -> neuron 2
    4: 3   # sea       -> neuron 3
}

# [0,4] => building and sea
tasks_definition = {
        0: [0, 4],
        1: [1, 3]

    }
    '''

In [ ]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Ensure the device is available
device = "cuda" if torch.cuda.is_available() else "cpu"

# Standard transformation for the model
my_transf = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Utility function to create loaders with label remapping
def create_mapped_loader(df, root_dir, label_map):
    # Copy dataframe to avoid modifying the original data
    df_mapped = df.copy()
    # Apply remapping for the 4-output classifier
    df_mapped['label'] = df_mapped['label'].map(label_map)

    # Use IntelDataset defined previously
    ds = IntelDataset(df_mapped, transform=my_transf, base_dir=root_dir)
    return DataLoader(ds, batch_size=batch_size, shuffle=False)

# Create the test loaders using variables and paths from the first phase
dict_test_loader = {
    0: create_mapped_loader(label_AB_test, path_img_AB_test, global_map),
    1: create_mapped_loader(label_CD_test, path_img_CD_test, global_map)
}

In [ ]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.cluster import KMeans

class EpisodicMemory:
    def __init__(self, max_memories_sample=1000):
        """
        max_memory_sample : the max images to keep for each past task
        """
        self.max_memories_sample = max_memories_sample
        self.memory_dataset = None   # No TensorDataset yet

    def _extract_features(self, model, dataset, device):
        """
        extract the features of the layer just before the last layer
        """
        # switch to eval mode
        model.eval()

        # We take as feature extractor the whole model except the last layer
        backbone = torch.nn.Sequential(*list(model.children())[:-1]).to(device)
        features_list = []
        temp_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

        with torch.no_grad():
            for images, _ in temp_loader:
                images = images.to(device)
                # the features go into the backbone to have the features
                features = backbone(images)
                features = features.view(features.size(0), -1)
                features_list.append(features.cpu().numpy())

        model.train()
        return np.concatenate(features_list, axis=0)

    def KMeans_sample_extractor(self, task_dataset, model=None, device='cuda'):
        """
        Select representative samples using KMeans clustering.
        """
        budget = min(self.max_memories_sample, len(task_dataset))

        if model is not None:
            print("Extraction of the features with KMeans")
            features = self._extract_features(model, task_dataset, device)

            print(f'We use KMeans with K={budget}')
            # Use of KMeans to create clusters
            kmeans = KMeans(n_clusters=budget, n_init='auto', random_state=42)
            cluster_labels = kmeans.fit_predict(features)
            # the centroids
            centroids = kmeans.cluster_centers_

            # Selection of the best image for each cluster
            selected_indices = []
            for i in range(budget):
                idx_in_cluster = np.where(cluster_labels == i)[0]
                if len(idx_in_cluster) > 0:
                    cluster_features = features[idx_in_cluster]
                    # We compute the distances
                    distances = np.linalg.norm(cluster_features - centroids[i], axis=1)
                    # The best indices
                    best_local_idx = np.argmin(distances)
                    selected_indices.append(idx_in_cluster[best_local_idx])

            # extraction of the tensors (images,labels) linked to these selected indices
            full_loader = DataLoader(task_dataset, batch_size=len(task_dataset), shuffle=False)
            all_images, all_labels = next(iter(full_loader))

            print("K-Means ended")
            return all_images[selected_indices], all_labels[selected_indices]

    def store_memories(self, task_dataset, method="random"):
        """
        extracts a random sample from the task that has just finished
        Method: “random” for the standard random method, “KMeans” for the K-means method
        """
        if method == "KMeans":
            images_sample, labels_sample = self.KMeans_sample_extractor(task_dataset, model, device)
        else:
            budget = min(self.max_memories_sample, len(task_dataset))
            # We create a temporary dataloader with batchsize=self.max_memories_sample
            temp_loader = DataLoader(task_dataset, batch_size=budget, shuffle=True)
            images_sample, labels_sample = next(iter(temp_loader))

        if self.memory_dataset is None:
            # Merge the images and the labels in a tensordataset
            self.memory_dataset = TensorDataset(images_sample, labels_sample)
        else:
            old_images, old_labels = self.memory_dataset.tensors
            # We merge the past and the present
            new_images = torch.cat([old_images, images_sample])
            new_labels = torch.cat([old_labels, labels_sample])
            self.memory_dataset = TensorDataset(new_images, new_labels)
            print(f"-> A-GEM Buffer updated. Total memory size: {len(self.memory_dataset)} images.")

    def get_reference_batch(self, batch_size):
        if self.memory_dataset is None:
            return None

        ref_loader = DataLoader(self.memory_dataset, batch_size=batch_size, shuffle=True)
        return next(iter(ref_loader))

In [ ]:
def train_agrem_step(model,optimizer,criterion,current_batch,batch_ref,device):
  """
  Train the model using A-GEM model for one step.
  model : the model used
  optimizer : the optimizer used
  current_batch : the batch (images,labels) selected to determine g
  batch_ref : the batch (images,labels) selected to determine g_ref
  """

  ## Initialisations

  current_imgs, current_labels=current_batch
  current_imgs, current_labels=current_imgs.to(device),current_labels.to(device)

  img_ref,label_ref=batch_ref
  img_ref,label_ref=img_ref.to(device),label_ref.to(device)

  #We calculate g
  optimizer.zero_grad()
  current_output=model(current_imgs)
  current_loss=criterion(current_output, current_labels)
  current_loss.backward()

  #selection of parameters that need a gradient
  list_of_grad1=[p.grad for p in model.parameters() if p.requires_grad ]

  # Transformation into a 1D vector to have the final one
  g=parameters_to_vector(list_of_grad1).clone()


# we calculate de g_ref
  optimizer.zero_grad()

  output_ref=model(img_ref)
  loss_ref=criterion(output_ref,label_ref)
  loss_ref.backward()

  list_of_grad2=[p.grad for p in model.parameters() if p.requires_grad ]
  g_ref=parameters_to_vector(list_of_grad2).clone()

  #dot product beteween g and g_ref
  dot_product=torch.dot(g,g_ref)

  #A-GEM  Projection
  if dot_product <0:
    terme_bas=torch.dot(g_ref,g_ref)
    coeff=dot_product / terme_bas

    # to calculate g - ( (g^t * g_ref) / (g_ref^T*g_ref) ) * g_ref
    g_proj = g - (coeff * g_ref )

    g = g_proj

  # --- MANUAL GRADIENT INJECTION ---
  # 1. Reset current gradients (which currently store g_ref from the backward pass)
  optimizer.zero_grad()

  index=0
  # 2. Iterate through all model parameters to re-assign the projected gradient
  for p in model.parameters():

    # Only update layers that are trainable (requires_grad = True)
    if p.requires_grad:

      # Get the number of elements in this specific layer/tensor
      num_params=p.numel()

      # Extract the corresponding slice ('tranche') from the global projected vector 'g'
      tranche=g[index:index + num_params]

      # Reshape the slice to match the layer's original shape and assign it to p.grad
      p.grad=tranche.view(p.shape).clone()

      # Increment the index to point to the next parameter slice in the vector
      index+=num_params

  # 3. Perform the optimization step using the newly injected projected gradients
  optimizer.step()

  return current_loss.item()

In [ ]:
def train_standard_step(model, optimizer,criterion,current_batch,device):
  """
  Train the model WITHOUT using A-GEM model for one step.
  model : the model used
  optimizer : the optimizer used
  current_batch : the batch (images,labels) selected to determine g
  Only used at the firts step
  """

  current_imgs,current_labels=current_batch
  # Reset the gradient
  optimizer.zero_grad()

  predictions=model(current_imgs)

  #we compute the loss
  loss=criterion(predictions,current_labels)

  loss.backward()
  optimizer.step()

  return loss.item()

In [ ]:
def evaluate_continual_eval(model, test_loaders, list_of_learned_task, device):
    """
    Evaluate the model on the test loaders for each task.
    """
    model.eval()
    results = {}  # Format: {id_task: {'acc': val, 'loss': val}}

    with torch.no_grad():
        for id_task in list_of_learned_task:
            specific_test_loader = test_loaders[id_task]
            total_correct = 0
            total_loss = 0.0
            total_samples = 0

            # get valid neurons for the task
            task_classes = tasks_definition[id_task]
            valid_neurons = [global_map[c] for c in task_classes]

            for images, labels in specific_test_loader:
                images, labels = images.to(device), labels.to(device)

                logits = model(images)

                # Masking neurons from other tasks
                # We create a mask filled with -1e9 (minus infinity)
                mask = torch.full_like(logits, -1e9)
                # We reactivate only the neurons of the current task
                mask[:, valid_neurons] = 0
                # Non-task neurons become ultra-negative
                masked_logits = logits + mask

                # Calculate loss using the global criterion
                loss = criterion(logits, labels)

                _, preds = torch.max(masked_logits, dim=1)
                total_correct += (preds == labels).sum().item()
                total_loss += loss.item() * labels.size(0)
                total_samples += labels.size(0)

            results[id_task] = {
                'acc': total_correct / total_samples,
                'loss': total_loss / total_samples
            }

    model.train()
    return results

In [ ]:
from tqdm import tqdm

def train_agem_global(model, optimizer, criterion, dict_test_loader, device='cuda'):
    """
    The final function to train the model with the A-GEM method (for all the steps)
    """

    # We create the buffer
    buffer_agem = EpisodicMemory(max_memories_sample=memory_size_per_task)

    # list of tasks already learned
    list_of_learned_task = []

    model.to(device)

    # Use the registry to access task-specific information
    for id_task in list_of_task:
        task_info = task_data_registry[id_task]
        current_df = task_info['train_df']
        current_path = task_info['train_path']

        print(f"\n--- Task {id_task} (Classes: {tasks_definition[id_task]}) ---")

        # transformation in Dataset -> DataLoader using variables from the environment
        current_dataset = IntelDataset(current_df, transform=my_transf, base_dir=current_path, label_map=global_map)
        current_loader = DataLoader(current_dataset, batch_size=batch_size, shuffle=True)

        for epoch in range(1, epochs_per_task + 1):
            model.train()
            for imgs, labels in tqdm(current_loader, desc=f"Epoch {epoch}"):
                imgs, labels = imgs.to(device), labels.to(device)

                # We get the reference batch (used for g_ref)
                batch_ref = buffer_agem.get_reference_batch(batch_size=batch_size)

                if batch_ref is not None:
                    train_agrem_step(model, optimizer, criterion, (imgs, labels), batch_ref, device)
                else:
                    # Because this is the first training => no knowledge to protect
                    train_standard_step(model, optimizer, criterion, (imgs, labels), device)

        # Select and add the samples to the buffer with the selected method
        buffer_agem.store_memories(current_dataset, method='random')
        list_of_learned_task.append(id_task)

        # We evaluate the model for all learned tasks
        results = evaluate_continual_eval(model, dict_test_loader, list_of_learned_task, device)
        print(f"\n Result after task {id_task} :")
        for t_id, metrics in results.items():
            print(f"   * Task {t_id} -> Precision: {metrics['acc']*100:.2f}% | Loss: {metrics['loss']:.4f}")

In [ ]:
from tqdm import tqdm

def train_agem_global(model, optimizer, criterion, dict_test_loader, device='cuda'):
  """
  The final function to train the model with the A-GEM method (for all the steps)
  """

  # We create the buffer
  buffer_agem = EpisodicMemory(max_memories_sample=memory_size_per_task)

  # list of tasks already learned
  list_of_learned_task = []

  model.to(device)

  # Use the registry to access task-specific information
  for id_task in list_of_task:
    task_info = task_data_registry[id_task]
    current_df = task_info['train_df']
    current_path = task_info['train_path']

    print(f"\n--- Task {id_task} (Classes: {tasks_definition[id_task]}) ---")

    # transformation in Dataset -> DataLoader using variables from the environment
    current_dataset = IntelDataset(current_df, transform=my_transf, base_dir=current_path, label_map=global_map)
    current_loader = DataLoader(current_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(1, epochs_per_task + 1):
      model.train()
      for imgs, labels in tqdm(current_loader, desc=f"Epoch {epoch}"):
        imgs, labels = imgs.to(device), labels.to(device)

        # We get the reference batch (used for g_ref)
        batch_ref = buffer_agem.get_reference_batch(batch_size=batch_size)

        if batch_ref is not None:
          train_agrem_step(model, optimizer, criterion, (imgs, labels), batch_ref, device)
        else:
          # Because this is the first training => no knowledge to protect
          train_standard_step(model, optimizer, criterion, (imgs, labels), device)

    # Select and add the samples to the buffer with the selected method
    buffer_agem.store_memories(current_dataset,method='random')
    list_of_learned_task.append(id_task)

    # We evaluate the model for all learned tasks
    results = evaluate_continual_eval(model, dict_test_loader, list_of_learned_task, device)
    print(f"\n Result after task {id_task} :")
    for t_id, metrics in results.items():
      print(f"   * Task {t_id} -> Precision: {metrics['acc']*100:.2f}% | Loss: {metrics['loss']:.4f}")

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils import parameters_to_vector
import os

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"


In [ ]:
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim

# 1. We clear the cache
torch.cuda.empty_cache()

# 2. Initialization of the pre-trained model
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 3. Replacing the last layer
model.fc = nn.Linear(model.fc.in_features, 4)

for param in model.parameters():
    param.requires_grad = True

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001, momentum=0.9)

# 5. Launch of global training
train_agem_global(model, optimizer, criterion, dict_test_loader, device)


--- Tâche 0 (Classes: [0, 4]) ---


Epoch 6: 100%|██████████| 140/140 [00:12<00:00, 10.81it/s]



 Result after task 0 :
   * Task 0 -> Precision: 99.58% | Loss: 0.0170

--- Tâche 1 (Classes: [1, 3]) ---


Epoch 6: 100%|██████████| 150/150 [00:25<00:00,  5.92it/s]


-> A-GEM Buffer updated. Total memory size: 600 images.

 Result after task 1 :
   * Task 0 -> Precision: 95.67% | Loss: 18.3517
   * Task 1 -> Precision: 99.60% | Loss: 0.0118
